In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

optuna.logging.set_verbosity(optuna.logging.WARNING)

ROOT = Path.cwd()
if not (ROOT / "jazz_harmony_ml_dataset.csv").exists():
    ROOT = ROOT.parent

jazz = pd.read_csv(ROOT / "jazz_harmony_ml_dataset.csv")
pop  = pd.read_csv(ROOT / "pop_harmony_dataset.csv")

# Combined dataset — only the columns used for training/lookup (no MIDI/Voicing)
data = pd.concat([
    jazz[["Key", "Chord_Progression", "ChordName", "Emotion", "Scale"]],
    pop[["Key", "Chord_Progression", "ChordName", "Emotion", "Scale"]],
], ignore_index=True)

print(f"Jazz: {len(jazz)} | Pop: {len(pop)} | Combined: {len(data)}")
print(f"Unique progressions: {data['Chord_Progression'].nunique()}")
print(f"Unique (Key, Prog) pairs: {data.drop_duplicates(['Key','Chord_Progression']).shape[0]}")
data.head()

/opt/miniconda3/envs/ml/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Jazz: 816 | Pop: 446 | Combined: 1262
Unique progressions: 80
Unique (Key, Prog) pairs: 650


,Key,Chord_Progression,ChordName,Emotion,Scale
0,Bb,V7-#IVm7b5-IVm7-IIIm7,F7-Em7b5-Ebm7-Dm7,Joyful,Ionian
1,A,V7-#IVm7b5-IVm7-IIIm7,E7-Ebm7b5-Dm7-Dbm7,Joyful,Ionian
2,B,V7-#IVm7b5-IVm7-IIIm7,F#7-Fm7b5-Em7-Ebm7,Fantasy,Ionian
3,Db,V7-#IVm7b5-IVm7-IIIm7,Ab7-Gm7b5-F#m7-Fm7,Depressive,Ionian
4,C,V7-#IVm7b5-IVm7-IIIm7,G7-F#m7b5-Fm7-Em7,Joyful,Ionian


In [2]:
le_emotion = LabelEncoder()
le_scale   = LabelEncoder()
le_prog    = LabelEncoder()
le_voicing = LabelEncoder()

le_emotion.fit(data["Emotion"])          # 13 classes
le_scale.fit(data["Scale"])              # 7 classes
le_prog.fit(data["Chord_Progression"])   # 80 classes (jazz 17 + pop 63, no overlap)
le_voicing.fit(jazz["Voicing"])          # 4 classes — jazz-only

# (Key, Progression) → ChordName
assert data.groupby(["Key", "Chord_Progression"])["ChordName"].nunique().max() == 1
chordname_lookup = (
    data.drop_duplicates(["Key", "Chord_Progression"])
    .set_index(["Key", "Chord_Progression"])["ChordName"]
    .to_dict()
)

# (Emotion, Scale) → list of unique (Key, Progression) pairs
# Used in generate_harmony to score all candidates simultaneously
_dedup = data.drop_duplicates(["Emotion", "Scale", "Key", "Chord_Progression"])
valid_pairs_lookup = {
    (em, sc): list(zip(grp["Key"], grp["Chord_Progression"]))
    for (em, sc), grp in _dedup.groupby(["Emotion", "Scale"])
}

# Valid scales per emotion (for input validation)
valid_scales = data.groupby("Emotion")["Scale"].apply(set).to_dict()

# Jazz progressions set — determines whether to return a voicing
jazz_progressions = set(jazz["Chord_Progression"])

print(f"Progression classes: {len(le_prog.classes_)}")
print(f"Voicing classes:     {len(le_voicing.classes_)}")
print(f"ChordName entries:   {len(chordname_lookup)}")
pairs_per_combo = [len(v) for v in valid_pairs_lookup.values()]
print(f"Pairs per (emotion, scale): min={min(pairs_per_combo)}  "
      f"avg={sum(pairs_per_combo)/len(pairs_per_combo):.1f}  "
      f"max={max(pairs_per_combo)}")

Progression classes: 80
Voicing classes:     4
ChordName entries:   650
Pairs per (emotion, scale): min=1  avg=10.7  max=91


In [3]:
# Progression: combined data; Voicing: jazz-only
X_all     = np.column_stack([le_emotion.transform(data["Emotion"]),
                               le_scale.transform(data["Scale"])])
y_prog    = le_prog.transform(data["Chord_Progression"])

X_jazz    = np.column_stack([le_emotion.transform(jazz["Emotion"]),
                               le_scale.transform(jazz["Scale"])])
y_voicing = le_voicing.transform(jazz["Voicing"])

X_tr_p, X_te_p, ytr_p, yte_p = train_test_split(
    X_all,  y_prog,    test_size=0.2, random_state=42, stratify=data["Emotion"])
X_tr_v, X_te_v, ytr_v, yte_v = train_test_split(
    X_jazz, y_voicing, test_size=0.2, random_state=42, stratify=jazz["Emotion"])

N_TRIALS = 50
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


def objective(trial):
    params = {
        "n_estimators":      trial.suggest_int("n_estimators", 50, 500, step=50),
        "max_depth":         trial.suggest_int("max_depth", 2, 32),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf":  trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features":      trial.suggest_categorical("max_features", ["sqrt", "log2", 1.0]),
        "random_state": 42,
    }
    s_p = cross_val_score(RandomForestClassifier(**params), X_tr_p, ytr_p,
                          cv=CV, scoring="accuracy", n_jobs=-1).mean()
    s_v = cross_val_score(RandomForestClassifier(**params), X_tr_v, ytr_v,
                          cv=CV, scoring="accuracy", n_jobs=-1).mean()
    return (s_p + s_v) / 2


study = optuna.create_study(direction="maximize", study_name="harmony_unified_rf")
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

best_params = {**study.best_params, "random_state": 42}
print(f"Best CV accuracy (avg): {study.best_value:.4f}")
print("Best params:", best_params)

clf_prog    = RandomForestClassifier(**best_params)
clf_voicing = RandomForestClassifier(**best_params)
clf_prog.fit(X_tr_p, ytr_p)
clf_voicing.fit(X_tr_v, ytr_v)

print(f"\nTraining complete.")
print(f"  Progression classes: {len(le_prog.classes_)}")
print(f"  Voicing classes:     {len(le_voicing.classes_)}")

  0%|          | 0/50 [00:00<?, ?it/s]/opt/miniconda3/envs/ml/lib/python3.11/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
Best trial: 0. Best value: 0.143122:   2%|▏         | 1/50 [00:01<01:29,  1.83s/it]/opt/miniconda3/envs/ml/lib/python3.11/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
Best trial: 0. Best value: 0.143122:   4%|▍         | 2/50 [00:02<01:01,  1.28s/it]/opt/miniconda3/envs/ml/lib/python3.11/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
Best trial: 2. Best value: 0.157052:   6%|▌         | 3/50 [00:04<01:00,  1.29s/it]/opt/miniconda3/envs/ml/lib/python3.11/site-packages/sklearn/model_selection/_split.py:813: UserWarni

Best CV accuracy (avg): 0.1722
Best params: {'n_estimators': 150, 'max_depth': 3, 'min_samples_split': 14, 'min_samples_leaf': 4, 'max_features': 1.0, 'random_state': 42}

Training complete.
  Progression classes: 80
  Voicing classes:     4


In [4]:
def top_k_acc(clf, X, y_true, k=3):
    proba = clf.predict_proba(X)
    top_k_cols   = proba.argsort(axis=1)[:, -k:]
    top_k_labels = clf.classes_[top_k_cols]   # col index → class label
    return sum(int(y_true[i] in top_k_labels[i]) for i in range(len(y_true))) / len(y_true)

K = 3
for name, clf, Xte, yte, le_p in [
    ("Progression", clf_prog,    X_te_p, yte_p, le_prog),
    ("Voicing",     clf_voicing, X_te_v, yte_v, le_voicing),
]:
    pred   = clf.predict(Xte)
    labels = np.unique(yte)
    n      = len(le_p.classes_)
    print(f"{name} ({n} classes):")
    print(f"  argmax={accuracy_score(yte, pred):.3f}  "
          f"top-{K}={top_k_acc(clf, Xte, yte, k=K):.3f}  "
          f"| random argmax=1/{n}={1/n:.3f}  random top-{K}={min(K,n)}/{n}={min(K,n)/n:.3f}")
    print(classification_report(yte, pred,
          labels=labels, target_names=le_p.classes_[labels], zero_division=0))

Progression (80 classes):
  argmax=0.142  top-3=0.328  | random argmax=1/80=0.013  random top-3=3/80=0.037
                       precision    recall  f1-score   support

             I-II-V-I       1.00      1.00      1.00         1
          I-III-IV-iv       0.00      0.00      0.00         3
                 I-IV       0.00      0.00      0.00         1
             I-IV-I-V       0.00      0.00      0.00         4
            I-IV-V-IV       0.00      0.00      0.00         3
            I-IV-V-vi       0.00      0.00      0.00         2
            I-IV-ii-V       0.00      0.00      0.00         2
          I-IV-iii-VI       0.00      0.00      0.00         1
          I-IV-iii-vi       0.00      0.00      0.00         4
            I-IV-vi-V       0.00      0.00      0.00         1
    I-IVsus-bVII-Vsus       0.00      0.00      0.00         1
             I-V-IV-I       0.00      0.00      0.00         2
           I-V-iii-IV       0.00      0.00      0.00         3
          

In [5]:
def _apply_temperature(probs: np.ndarray, temperature: float) -> np.ndarray:
    """T→0 argmax  |  T=1 unchanged  |  T>1 flatter/exploratory"""
    if temperature <= 0:
        out = np.zeros_like(probs, dtype=float)
        out[np.argmax(probs)] = 1.0
        return out
    if abs(temperature - 1.0) < 1e-9:
        return probs / probs.sum()
    scaled = np.power(probs, 1.0 / temperature)
    return scaled / scaled.sum()


def generate_harmony(
    emotion: str,
    scale: str | None = None,
    sample: bool = True,
    temperature: float = 1.0,
    random_state: int = 42,
) -> dict:
    """
    emotion     : one of the 13 emotion labels
    scale       : one of 7 scale labels, or None to auto-sample
    temperature : T→0 conservative  T=1 model dist  T>1 exploratory
    """
    if emotion not in le_emotion.classes_:
        raise ValueError(f"Unknown emotion {emotion!r}. Valid: {sorted(le_emotion.classes_)}")

    rng = np.random.default_rng(random_state)

    # 1. Resolve scale ────────────────────────────────────────────────────────
    if scale is None:
        sc_counts = data[data["Emotion"] == emotion]["Scale"].value_counts()
        sc_w = _apply_temperature(
            sc_counts.values.astype(float) / sc_counts.values.sum(), temperature
        )
        scale = sc_counts.index[rng.choice(len(sc_counts), p=sc_w)] if sample                 else sc_counts.index[0]
    elif scale not in valid_scales[emotion]:
        raise ValueError(
            f"Scale {scale!r} not valid for {emotion!r}. "
            f"Valid: {sorted(valid_scales[emotion])}"
        )

    # 2. Score every valid (key, progression) pair for this (emotion, scale) ─
    #    All pairs compete simultaneously — temperature controls the spread.
    pairs = valid_pairs_lookup[(emotion, scale)]
    x = np.array([[le_emotion.transform([emotion])[0],
                   le_scale.transform([scale])[0]]])

    probs       = clf_prog.predict_proba(x)[0]
    col_of      = {c: col for col, c in enumerate(clf_prog.classes_)}

    pair_scores = np.array([
        probs[col_of[le_prog.transform([prog])[0]]]
        if le_prog.transform([prog])[0] in col_of else 1e-10
        for _, prog in pairs
    ], dtype=float)
    pair_scores  = _apply_temperature(pair_scores / pair_scores.sum(), temperature)

    # 3. Sample (key, progression) ────────────────────────────────────────────
    idx        = rng.choice(len(pairs), p=pair_scores) if sample else int(np.argmax(pair_scores))
    key, progression = pairs[idx]

    # 4. Voicing (jazz progressions only) ─────────────────────────────────────
    voicing = None
    if progression in jazz_progressions:
        valid_voicings = set(jazz.loc[jazz["Key"] == key, "Voicing"])
        probs_v = clf_voicing.predict_proba(x)[0]
        col_v   = {c: col for col, c in enumerate(clf_voicing.classes_)}
        v_idx   = [i for i, lbl in enumerate(le_voicing.classes_)
                   if lbl in valid_voicings and i in col_v]
        v_scores = np.array([probs_v[col_v[i]] for i in v_idx], dtype=float)
        v_scores = _apply_temperature(v_scores / v_scores.sum(), temperature)
        vi = rng.choice(len(v_idx), p=v_scores) if sample else int(np.argmax(v_scores))
        voicing = le_voicing.inverse_transform([v_idx[vi]])[0]

    result = {
        "Emotion":           emotion,
        "Scale":             scale,
        "Key":               key,
        "Chord_Progression": progression,
        "ChordName":         chordname_lookup[(key, progression)],
    }
    if voicing:
        result["Voicing"] = voicing
    return result


# Validation
lookup_ok = all(
    chordname_lookup[(r.Key, r.Chord_Progression)] == r.ChordName
    for r in data.itertuples()
)
print(f"ChordName lookup covers all rows: {lookup_ok}")
result = generate_harmony("Joyful", sample=False)
for k, v in result.items():
    print(f"  {k}: {v}")

ChordName lookup covers all rows: True
  Emotion: Joyful
  Scale: Ionian
  Key: A
  Chord_Progression: IVmaj7-V7-Imaj7
  ChordName: Dmaj7-E7-Amaj7
  Voicing: Drop 2+4


In [6]:
# Temperature demo: T→0 conservative | T=1 model dist | T>1 exploratory
demo_emotion = "Joyful"
print(f"Emotion: {demo_emotion}\n")

for temp in [0.3, 1.0, 2.0]:
    print(f"Temperature = {temp}")
    seen = set()
    for i in range(8):
        out = generate_harmony(demo_emotion, sample=True, temperature=temp, random_state=i)
        voicing_str = f" | {out['Voicing']}" if out.get("Voicing") else ""
        tag = f"{out['Key']} | {out['Scale']} | {out['Chord_Progression']}"
        print(f"  [{i+1}] {tag}{voicing_str} → {out['ChordName']}")
        seen.add(tag)
    print(f"  unique combinations: {len(seen)}/8\n")

Emotion: Joyful

Temperature = 0.3
  [1] Bb | Ionian | V7-bIIImaj7 | Drop 2 → F7-Dbmaj7
  [2] Bb | Ionian | IVmaj7-V7-Imaj7 | Drop 2 → Ebmaj7-F7-Bbmaj7
  [3] A | Ionian | V7-bIIImaj7 | Four-Way Close → E7-Cmaj7
  [4] C | Ionian | V7-VIm7 | Four-Way Close → G7-Am7
  [5] A | Ionian | V7-bVImaj7 | Four-Way Close → E7-Fmaj7
  [6] A | Ionian | IVmaj7-IVm6-Imaj7 | Drop 3 → Dmaj7-Dm6-Amaj7
  [7] C | Ionian | V7-bIIImaj7 | Drop 2+4 → G7-Ebmaj7
  [8] A | Ionian | IVmaj7-V7-Imaj7 | Four-Way Close → Dmaj7-E7-Amaj7
  unique combinations: 8/8

Temperature = 1.0
  [1] Bb | Ionian | V7-bIImaj7 | Drop 2 → F7-Bmaj7
  [2] Bb | Ionian | IV-iii-vi-II → Eb-Dm-Gm-C
  [3] A | Ionian | V7-bIImaj7 | Four-Way Close → E7-Bbmaj7
  [4] C | Ionian | V7-bIIImaj7 | Four-Way Close → G7-Ebmaj7
  [5] A | Aeolian | bVImaj7-bVII7-Imaj7 | Four-Way Close → Fmaj7-G7-Amaj7
  [6] Bb | Ionian | vi-IV-I-V → Gm-Eb-Bb-F
  [7] Bb | Ionian | V7-bVImaj7 | Drop 2+4 → F7-F#maj7
  [8] A | Ionian | IV-V-I → D-E-A
  unique combinations: 8